In [ ]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

In [ ]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

In [ ]:
!pip install -q pydantic-ai-slim openai

In [ ]:
pip install "pydantic-ai-slim[openai]"

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

In [ ]:
pip install "pydantic-ai-slim[mcp]"

In [ ]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


In [ ]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [ ]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

In [ ]:
!pip install -q mcp-server-time

In [ ]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

In [ ]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

In [ ]:
# Install Node.js 20 via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt install -y nodejs

In [ ]:
!node -v && npm -v && npx --version

In [ ]:
pip install pandas openpyxl

In [ ]:
import pandas as pd
# Load Excel file
df = pd.read_excel("Service_Contact_Repository_training_data.xlsx", engine="openpyxl")
# Preview
print(df.head())

In [ ]:
print(df.columns)

In [ ]:
# Convert to list of dicts
data = [{"instruction": q, "output": a} for q, a in zip(df["Third Party Name"], df["Latest contact"])]


In [ ]:

# Save to JSONL
import json
with open("train.jsonl", "w", encoding="utf-8") as f:
    for row in data:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


In [ ]:
#Load dataset for training
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl")

In [ ]:
#Prepare Qwen model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2-1.5B"  # example model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ✅ Tokenization function
def tokenize(batch):
    texts = [instr + "\n" + out for instr, out in zip(batch["instruction"], batch["output"])]
    return tokenizer(texts, truncation=True)

# Apply tokenization
tokenized = dataset.map(tokenize, batched=True)

# ✅ Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# ✅ Data collator for causal language modeling (Qwen is causal LM, not masked LM)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ✅ Training arguments with memory‑friendly settings
training_args = TrainingArguments(
    output_dir="./qwen-finetuned",
    per_device_train_batch_size=1,       # smaller batch size
    gradient_accumulation_steps=4,       # accumulate gradients to simulate larger batch
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="epoch",
                      # mixed precision (saves memory)
    optim="adamw_torch"                  # efficient optimizer
)

# ✅ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    data_collator=data_collator
)

# ✅ Start training
trainer.train()
